In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\hp\Desktop\Amazon\CSV Files\amazon_india_2022.csv")

In [3]:
df.shape

(132660, 34)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132660 entries, 0 to 132659
Data columns (total 34 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   transaction_id          132660 non-null  object 
 1   order_date              132660 non-null  object 
 2   customer_id             132660 non-null  object 
 3   product_id              132660 non-null  object 
 4   product_name            132660 non-null  object 
 5   category                132660 non-null  object 
 6   subcategory             132660 non-null  object 
 7   brand                   132660 non-null  object 
 8   original_price_inr      132660 non-null  object 
 9   discount_percent        132660 non-null  float64
 10  discounted_price_inr    132660 non-null  float64
 11  quantity                132660 non-null  int64  
 12  subtotal_inr            132660 non-null  float64
 13  delivery_charges        122044 non-null  float64
 14  final_amount_inr    

In [5]:
import numpy as np

df.replace("", np.nan, inplace=True)


In [6]:
dfc = df.copy()

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [9]:
import pandas as pd

dfc['order_date'] = (
    dfc['order_date']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', '', regex=True)
)

dfc['order_date'] = pd.to_datetime(
    dfc['order_date'],
    dayfirst=True,
    errors='coerce'
)
dfc['order_date'] = dfc['order_date'].dt.strftime('%Y-%m-%d')


C:\Users\hp\AppData\Local\Temp\ipykernel_16560\570715499.py:10: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  dfc['order_date'] = pd.to_datetime(


In [11]:
dfc['order_date'].head(20)

0     2022-01-22
1     2022-01-11
2            NaN
3     2022-01-27
4     2022-01-16
5     2022-01-02
6     2022-01-16
7     2022-01-15
8     2022-01-01
9     2022-01-25
10    2022-01-26
11    2022-01-31
12    2022-01-20
13    2022-01-06
14    2022-01-17
15    2022-01-03
16    2022-01-22
17    2022-01-12
18    2022-01-22
19    2022-01-12
Name: order_date, dtype: object

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees.

In [12]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
        .astype(str)                      
        .str.replace('₹', '', regex=False) 
        .str.replace(',', '', regex=False)
        .str.replace('Rs ', '', regex=False)
        .str.strip()                
)

dfc['original_price_inr'] = pd.to_numeric(
    dfc['original_price_inr']
)


Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.

In [13]:
import re

def parse_rating(r):
    if pd.isna(r):
        return np.nan

    if '/' in r:
        a, b = r.split('/')
        return float(a) / float(b) * 5

    m = re.search(r'\d+\.?\d*', r)
    return float(m.group()) if m else np.nan


dfc['customer_rating'] = dfc['customer_rating'].apply(parse_rating)

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.

In [14]:
dfc['customer_city'] = (
    dfc['customer_city']
    .str.lower()
    .str.strip()
)
 
city_map = {
    'bangalore': 'Bengaluru',
    'bengaluru': 'Bengaluru',
    'bangalore/bengaluru': 'Bengaluru',
    'bengalore' : 'Bengaluru',
    'Bengaluru' : 'banglore',
    
    'mumbai': 'Mumbai',
    'bombay': 'Mumbai',
    'mumbai/bombay': 'Mumbai',
    'mumba' : 'Mumbai',
    'calcutta' : 'kolkata',

    'delhi': 'Delhi',
    'new delhi': 'Delhi',
    'delhi/new delhi': 'Delhi',
    'delhi NCR' : 'Delhi',
    'delhi ncr' : 'Delhi',

    'chenai' : 'chennai',
    'madras' : 'chennai'
}

dfc['customer_city'] = dfc['customer_city'].replace(city_map)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [15]:
import numpy as np

bool_cols = ['is_prime_member', 'is_prime_eligible', 'is_festival_sale']

for col in bool_cols:
    dfc[col] = dfc[col].replace(['', ' ', 'NA', 'N/A', None, 'None'], np.nan)


bool_map = {
    True: True,
    'True': True,
    'true': True,
    'Yes': True,
    'yes': True,
    'Y': True,
    'y': True,
     1: True,
    
     False: False,
    'False': False,
    'false': False,
    'No': False,
    'no': False,
    'N': False,
    'n': False,
     0: False
}


for col in bool_cols:
    dfc[col] = dfc[col].map(bool_map)

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.

In [16]:
dfc.columns = dfc.columns.str.strip()

category_map = {
    'electronics': 'Electronics',
    'ELECTRONICS': 'Electronics',
    'electronics & accessories': 'Electronics',
    'Electronicss': 'Electronics',
    'Electronics & Accessories': 'Electronics',
    'Electronic': 'Electronics',
    'clothing': 'Fashion'
}

dfc['category'] = dfc['category'].replace(category_map)

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [17]:
import numpy as np

days_map = {
    'Express': '0',
    'Same Day': '0',
    '-1': 'None',
    '1-2 days': '2'
}

dfc['delivery_days'] = dfc['delivery_days'].replace(days_map)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [19]:
dup_cols = [
    "customer_id",
    "product_id",
    "order_date",
    "final_amount_inr"
]

price_cols = [
    "original_price_inr",
    "discounted_price_inr",
    "subtotal_inr",
    "final_amount_inr"
]

dfc["dup_count"] = (
    dfc.groupby(dup_cols)["transaction_id"]
      .transform("count")
)


dfc["price_identical"] = (
    dfc.groupby(dup_cols)[price_cols]
      .transform("nunique")
      .max(axis=1) == 1
)


dfc["is_high_value"] = dfc["final_amount_inr"] > 5000
dfc["is_bulk_customer"] = dfc["customer_spending_tier"].isin(["Premium"])
dfc["is_bulk_quantity"] = dfc["quantity"] > 1

dfc["is_duplicate_candidate"] = dfc["dup_count"] > 1


In [20]:
df_deduped = dfc[dfc["is_duplicate_candidate"]].drop_duplicates(subset=dup_cols, keep="first")


In [21]:
print("Rows deleted:", (df_deduped))

Rows deleted:            transaction_id  order_date         customer_id   product_id  \
452     TXN_2022_00000453  2022-01-02  CUST_2022_00023792  PROD_000011   
668     TXN_2022_00000669  2022-01-21  CUST_2022_00009112  PROD_000486   
1515    TXN_2022_00001516  2022-01-18  CUST_2018_00024264  PROD_001936   
1926    TXN_2022_00001927  2022-01-11  CUST_2022_00014977  PROD_000790   
2349    TXN_2022_00002350  2022-01-29  CUST_2021_00011420  PROD_001947   
...                   ...         ...                 ...          ...   
129809  TXN_2022_00129810  2022-12-03  CUST_2019_00015412  PROD_001591   
129897  TXN_2022_00129898  2022-12-09  CUST_2016_00007995  PROD_001918   
129921  TXN_2022_00129922  2022-12-24  CUST_2022_00029093  PROD_000584   
130525  TXN_2022_00130526  2022-12-06  CUST_2022_00029219  PROD_000238   
131840  TXN_2022_00131841  2022-12-18  CUST_2020_00019981  PROD_000176   

                          product_name     category  subcategory    brand  \
452     Apple iPhone

In [22]:
len(dfc)

132660

In [23]:
cols_to_drop = [
    "dup_count",
    "price_identical",
    "is_high_value",
    "is_bulk_customer",
    "is_bulk_quantity",
    "is_duplicate_candidate"
]

dfc = dfc.drop(columns=cols_to_drop)

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [24]:
import numpy as np

dfc["product_median_price"] = (
    dfc.groupby("product_id")["final_amount_inr"]
      .transform("median")
)

dfc["price_outlier"] = (
    dfc["final_amount_inr"] > 50 * dfc["product_median_price"]
)

dfc.loc[dfc["price_outlier"], "final_amount_inr"] /= 100
dfc.loc[dfc["price_outlier"], "discounted_price_inr"] /= 100
dfc.loc[dfc["price_outlier"], "original_price_inr"] /= 100

dfc["subtotal_inr"] = dfc["discounted_price_inr"] * dfc["quantity"]
dfc["final_amount_inr"] = dfc["subtotal_inr"] + dfc["delivery_charges"].fillna(0)

dfc["price_corrected_flag"] = dfc["price_outlier"]

dfc.drop(columns=["product_median_price"], inplace=True)

corrected_rows = dfc[dfc["price_corrected_flag"]]

print(corrected_rows)


Empty DataFrame
Columns: [transaction_id, order_date, customer_id, product_id, product_name, category, subcategory, brand, original_price_inr, discount_percent, discounted_price_inr, quantity, subtotal_inr, delivery_charges, final_amount_inr, customer_city, customer_state, customer_tier, customer_spending_tier, customer_age_group, payment_method, delivery_days, delivery_type, is_prime_member, is_festival_sale, festival_name, customer_rating, return_status, order_month, order_year, order_quarter, product_weight_kg, is_prime_eligible, product_rating, price_outlier, price_corrected_flag]
Index: []

[0 rows x 36 columns]


In [25]:
dfc = dfc.drop(
    columns=[
        "price_outlier",
        "price_corrected_flag"       
    ]
)



Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [26]:
dfc["payment_method"] = (
    dfc["payment_method"]
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.strip()
)

payment_map = {
    "UPI": "UPI",
    "PHONEPE": "UPI",
    "GOOGLEPAY": "UPI",
    "GPAY": "UPI",
    "PAYTM": "UPI",

    "CREDIT CARD": "Credit Card",
    "CREDIT_CARD": "Credit Card",
    "CC": "Credit Card",

    "DEBIT CARD": "Debit Card",
    "DC": "Debit Card",

    "COD": "Cash on Delivery",
    "CASH ON DELIVERY": "Cash on Delivery",

    "NET BANKING": "Net Banking"
}

dfc["payment_method"] = dfc["payment_method"].replace(payment_map)

In [27]:
dfc['original_price_inr'] = (
    dfc['original_price_inr']
    .astype(str)
    .str.replace('-', '', regex=False)
    .astype(float)
)


In [28]:
dfc["payment_method"].unique()

array(['UPI', 'Net Banking', 'Cash on Delivery', 'BNPL', 'WALLET',
       'Credit Card', 'Debit Card'], dtype=object)

In [29]:
import pandas as pd

columns = [
    "transaction_id","order_date","customer_id","product_id","product_name",
    "category","subcategory","brand","original_price_inr","discount_percent",
    "discounted_price_inr","quantity","subtotal_inr"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

transaction_id: ['TXN_2022_00000001', 'TXN_2022_00000002', 'TXN_2022_00000003', 'TXN_2022_00000004', 'TXN_2022_00000005', 'TXN_2022_00000006', 'TXN_2022_00000007', 'TXN_2022_00000008', 'TXN_2022_00000009', 'TXN_2022_00000010', 'TXN_2022_00000011', 'TXN_2022_00000012', 'TXN_2022_00000013', 'TXN_2022_00000014', 'TXN_2022_00000015', 'TXN_2022_00000016', 'TXN_2022_00000017', 'TXN_2022_00000018', 'TXN_2022_00000019', 'TXN_2022_00000020', 'TXN_2022_00000021', 'TXN_2022_00000022', 'TXN_2022_00000023', 'TXN_2022_00000024', 'TXN_2022_00000025', 'TXN_2022_00000026', 'TXN_2022_00000027', 'TXN_2022_00000028', 'TXN_2022_00000029', 'TXN_2022_00000030', 'TXN_2022_00000031', 'TXN_2022_00000032', 'TXN_2022_00000033', 'TXN_2022_00000034', 'TXN_2022_00000035', 'TXN_2022_00000036', 'TXN_2022_00000037', 'TXN_2022_00000038', 'TXN_2022_00000039', 'TXN_2022_00000040', 'TXN_2022_00000041', 'TXN_2022_00000042', 'TXN_2022_00000043', 'TXN_2022_00000044', 'TXN_2022_00000045', 'TXN_2

In [26]:
import pandas as pd

columns = [
    "delivery_charges",
    "final_amount_inr","customer_city","customer_state","customer_tier",
    "customer_spending_tier","customer_age_group","payment_method","delivery_days",
    "delivery_type","is_prime_member","is_festival_sale",
    "festival_name"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

delivery_charges: ['0.0', '40.0']

final_amount_inr: ['10000.7', '100008.66', '100008.93', '100014.44', '100017.68', '100019.07', '10002.36', '100022.83', '100022.96', '10003.27', '100031.16', '100033.56', '100034.93', '10004.03', '100042.2', '100044.91', '100047.36000000002', '100048.72', '100048.98', '10005.02', '10005.04', '10005.67', '10005.68', '100056.93', '100059.42', '10006.1', '100062.48', '100063.08', '100066.5', '100068.24', '100073.14', '100073.68', '100076.58', '100077.07', '100078.24', '100081.52', '100083.73', '100083.97', '100086.31', '100088.67', '10009.56', '100102.41', '100106.55', '10012.21', '100120.75', '100124.12', '100127.28', '100128.12', '100128.51', '100134.54000000001', '100134.66', '10014.34', '100140.27', '100141.54', '100144.3', '100153.93', '100156.3', '100157.64', '100163.13', '100164.51', '100165.62', '100168.3', '10017.32', '100173.8', '100176.5', '100179.75', '100183.23000000001', '100183.53', '100186.52', '100189.29',

In [27]:
import pandas as pd

columns = [    "customer_rating","return_status","order_month","order_year","order_quarter",
    "product_weight_kg","is_prime_eligible","product_rating"
]
unique_values = {}

for col in columns:
    if col in dfc.columns:
        unique_values[col] = sorted(dfc[col].dropna().astype(str).unique())
    else:
        unique_values[col] = []

print("Unique values for each column:\n")
for col, values in unique_values.items():
    print(f"{col}: {values}\n")


Unique values for each column:

customer_rating: ['3.0', '3.5', '4.0', '4.5', '5.0']

return_status: ['Cancelled', 'Delivered', 'Returned']

order_month: ['1', '10', '11', '12', '2', '3', '4', '5', '6', '7', '8', '9']

order_year: ['2021']

order_quarter: ['1', '2', '3', '4']

product_weight_kg: ['0.03', '0.04', '0.05', '0.06', '0.07', '0.08', '0.09', '0.1', '0.11', '0.12', '0.13', '0.14', '0.15', '0.16', '0.17', '0.18', '0.19', '0.2', '0.21', '0.22', '0.23', '0.24', '0.25', '0.27', '0.28', '0.29', '0.3', '0.31', '0.32', '0.33', '0.34', '0.35', '0.36', '0.37', '0.38', '0.39', '0.4', '0.41', '0.42', '0.43', '0.44', '0.45', '0.46', '0.47', '0.48', '0.49', '0.5', '0.51', '0.52', '0.53', '0.54', '0.55', '0.56', '0.57', '0.58', '0.59', '0.61', '0.62', '0.63', '0.64', '0.65', '0.66', '0.67', '0.68', '0.69', '0.7', '0.71', '0.72', '0.73', '0.75', '0.78', '1.2', '1.21', '1.24', '1.29', '1.31', '1.33', '1.37', '1.39', '1.4', '1.46', '1.48', '1.5', '1.51', '1.55', '1.57', '1.58', '1.62', '1.63',

In [30]:
print(len(dfc.columns))
print(dfc.columns.tolist())


34
['transaction_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'subcategory', 'brand', 'original_price_inr', 'discount_percent', 'discounted_price_inr', 'quantity', 'subtotal_inr', 'delivery_charges', 'final_amount_inr', 'customer_city', 'customer_state', 'customer_tier', 'customer_spending_tier', 'customer_age_group', 'payment_method', 'delivery_days', 'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name', 'customer_rating', 'return_status', 'order_month', 'order_year', 'order_quarter', 'product_weight_kg', 'is_prime_eligible', 'product_rating']


In [31]:
dfc[dfc.duplicated()]

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating


In [32]:
import pandas as pd

decimal_cols = dfc.select_dtypes(include=['float', 'float64']).columns

dfc[decimal_cols] = dfc[decimal_cols].round(2)
print(decimal_cols)


Index(['original_price_inr', 'discount_percent', 'discounted_price_inr',
       'subtotal_inr', 'delivery_charges', 'final_amount_inr',
       'customer_rating', 'product_weight_kg', 'product_rating'],
      dtype='object')


In [34]:
dfc.to_csv(r"C:\Users\hp\Desktop\Amazon\CSV_Clean_Files\amazon_india_2022_clean.csv",header='infer',index=False)

In [33]:
dfc

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2022_00000001,2022-01-22,CUST_2022_00015913,PROD_000527,Samsung Galaxy A50 64GB White,Electronics,Smartphones,Samsung,26992.07,41.06,...,True,Republic Day Sale,4.0,Delivered,1,2022,1,0.20,NaN,4.2
1,TXN_2022_00000002,2022-01-11,CUST_2022_00021843,PROD_000979,Samsung Galaxy S22+ 128GB White,Electronics,Smartphones,Samsung,80105.77,0.00,...,False,NaN,NaN,Delivered,1,2022,1,0.24,False,3.3
2,TXN_2022_00000003,NaN,CUST_2019_00043917,PROD_001909,Xiaomi Fitness Band Deluxe,Electronics,Smart Watch,Xiaomi,27516.92,0.00,...,False,NaN,4.0,Delivered,1,2022,1,0.05,True,4.2
3,TXN_2022_00000004,2022-01-27,CUST_2022_00011696,PROD_000284,Xiaomi Mi A1 32GB White,Electronics,Smartphones,Xiaomi,38018.11,0.00,...,NaN,NaN,4.0,Delivered,1,2022,1,0.24,True,4.6
4,TXN_2022_00000005,2022-01-16,CUST_2022_00030756,PROD_000556,OnePlus OnePlus 7T 64GB Gold,Electronics,Smartphones,OnePlus,87743.24,0.00,...,False,NaN,NaN,Delivered,1,2022,1,0.18,True,3.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
132655,TXN_2022_00027248_DUP,2022-03-16,CUST_2022_00011638,PROD_000972,Samsung Galaxy S22 128GB White,Electronics,Smartphones,Samsung,75294.24,7.30,...,False,NaN,NaN,Delivered,3,2022,1,0.23,True,4.2
132656,TXN_2022_00010369_DUP,2022-01-27,CUST_2022_00036128,PROD_001568,ASUS ThinkPad 4GB RAM Silver,Electronics,Laptops,ASUS,76375.15,20.34,...,False,NaN,5.0,Delivered,1,2022,1,2.69,True,4.5
132657,TXN_2022_00053894_DUP,NaN,CUST_2022_00001209,PROD_001945,Garmin Fitness Band,Electronics,Smart Watch,Garmin,42642.37,62.52,...,True,Back to School,5.0,Delivered,6,2022,2,0.05,False,3.3
132658,TXN_2022_00051412_DUP,NaN,CUST_2016_00009941,PROD_000529,Samsung Galaxy A50 64GB Blue,Electronics,Smartphones,Samsung,17093.64,38.66,...,True,Back to School,4.0,Delivered,6,2022,2,0.18,True,3.7
